# Master's Thesis Evaluation & Experimental Results
## Human Oversight and Human-in-the-Loop Control Engine for Autonomous AI Agents

This interactive notebook provides the statistical analysis, evaluation metrics, and publication-ready figures for the experimental chapter of the Master's thesis.

### Objectives Evaluated:
1. **Task Effectiveness & Gap Analysis**: Technical completion rate vs. Semantic correctness.
2. **Oversight Quality**: Escalation confusion matrix, Precision, Recall, F1-Score, and Human Intervention Rate.
3. **Pillar Breakdown**: Distribution of triggers across Evidence, Trajectory, Semantic Judge, and Static Policy pillars.
4. **System Overhead & Latency**: Actor turns, tool calls, governance passes, and human review duration.
5. **Ablation Study**: Comparative assessment across different oversight configurations.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project modules are importable
project_root = Path("..").resolve() if Path("..").resolve().name == "tesi" else Path(".").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

from notebooks.thesis_plotting import (
    setup_thesis_style,
    load_evaluation_data,
    create_sample_evaluation_data,
    plot_technical_vs_semantic_gap,
    plot_escalation_confusion_matrix,
    plot_governance_pillars_and_reasons,
    plot_overhead_and_efficiency,
    plot_ablation_comparison,
    export_all_thesis_figures,
)

setup_thesis_style()
print("✓ Thesis Evaluation environment initialized successfully.")

## 1. Ingestion & Data Loading
Load the evaluated runs directly from the SQLite database `artifacts/evaluation/layer5_evaluation.sqlite3`.
If the database is not yet populated, a synthetic benchmark dataset is loaded automatically for immediate demonstration.

In [ ]:
db_path = project_root / "artifacts" / "evaluation" / "layer5_evaluation.sqlite3"
data = load_evaluation_data(db_path)

if not data or data.get("runs", pd.DataFrame()).empty:
    print("ℹ No runs found in SQLite database. Generating demonstration benchmark dataset...")
    data = create_sample_evaluation_data()
else:
    print(f"✓ Successfully loaded {len(data['runs'])} runs from SQLite database.")

runs_df = data["runs"]
gov_df = data["governance_events"]
reviews_df = data["human_reviews"]
ops_df = data["external_operations"]

print(f"• Total Runs: {len(runs_df)}")
print(f"• Governance Passes: {len(gov_df)}")
print(f"• Human Reviews: {len(reviews_df)}")
print(f"• External Operations: {len(ops_df)}")

runs_df.head(5)

## 2. Technical Completion vs. Semantic Correctness (Gap Analysis)
A core claim in the thesis is that an agent reaching technical termination (`completed`) does not necessarily satisfy the goal. The chart below highlights this gap.

In [ ]:
fig1 = plot_technical_vs_semantic_gap(runs_df)
plt.show()

## 3. Human Oversight Confusion Matrix & Escalation Performance
Evaluating the decision engine (OPA + Pilastri + Semantic Judge):
- **True Positive**: Review was required and legitimately needed (risk prevented).
- **False Positive**: Review was required but was unnecessary (over-cautious policy).
- **False Negative**: Review was omitted on an unsafe or erroneous candidate (safety failure).
- **True Negative**: Autonomy was safely granted.

In [ ]:
fig2 = plot_escalation_confusion_matrix(gov_df)
plt.show()

## 4. Governance Pillars & Reason Code Breakdown
Analyzing which pillar of the oversight architecture prompted escalations and policy directives.

In [ ]:
fig3 = plot_governance_pillars_and_reasons(gov_df)
plt.show()

## 5. Computational Overhead & Review Latency
Distribution of actor turns, tool interactions, governance cycles, and operator response times.

In [ ]:
fig4 = plot_overhead_and_efficiency(runs_df, reviews_df)
plt.show()

## 6. Ablation Study & Configuration Comparison
Comparing the baseline multi-pillar system against ablated baselines (e.g. without Semantic Judge or Static-only).

In [ ]:
fig5 = plot_ablation_comparison(runs_df)
plt.show()

## 7. Export High-Resolution Figures for LaTeX
Saves all figures in vector PDF (`.pdf`) and 300 DPI PNG (`.png`) for direct inclusion in the LaTeX thesis document.

In [ ]:
figures_out_dir = project_root / "figures" / "thesis"
exported_files = export_all_thesis_figures(data, figures_out_dir)

print(f"✓ Successfully generated {len(exported_files)} figure files in: {figures_out_dir}\n")
for f in exported_files:
    print(f"  • {f.name}")